**Eurécom**

**Digital Communications**

Lab Session I October 20th 2025

# Receiver for NR Primary Synchronization Signals (PSS)

The goal of this lab session is to investigate the receivers for the PSS signals used 3GPP-NR systems. These are the most basic signals that a terminal must detect prior to initiating a connection with the basestation. This detection is also the first step in the so-called *cell search* procedure. The following illustration shows the system concept of the PSS signal which is the first component of the *Synchronization Signal Block (SSB)*. 
<center>
    <figure>
    <img src="pss_lab1.png" width="400" height="400">
    <figcaption>Illustration of outdoor deployment with PSS Signals</figcaption>
    </figure>
</center>
There are three basestations (gNodeB) in the vicinity of the user equipment (UE). Depending on the position of the UE the received signal during the PSS period will contain different amounts of energy corresponding to the 3 basestations. The PSS signal is an indicator firstly of the proportion of received energy from a particular basestation and allows the UE to select the right one for establishing a connection with the network. Secondly it will allow for time and frequency synchronization with the network. This lab session will explore the various detection procedures related to the PSS signal.

You are to commit and push a completed version of this notebook to your own branch on gitlab in the Digicom2025 group including answers to all the questions outlined here along with corresponding python code and figures embedded. Although we will work in the classroom, you are expected to work on your own time. The name of the branch should be "LabSession1_names", where "names" are the people that contributed to the work.

## PSS signals

We are concerned with complex baseband equivalent transmit signals of
the form 

$$\begin{equation*}\tilde{s}_{\mathrm{PSS},i}(n) = \sum_{k}x_i(n-kN), i=0,1,2\end{equation*}$$ 

where $N$ is the periodicity of the PSS signals which is $10^{-2} f_s$ samples and $f_s$ is the sampling frequency. The PSS is part of a multi-component signal called the SSB (Synchronization Signal Block) and there can be more than one in a period of 5ms depending on the antenna configuration, but here we will assume a simple scenario with a single SSB and therefore a single PSS as described below. $i$ is the index of the transmitted PSS signal and can have one of three values. In the context of this lab session, we will assume that the
receiver uses a sampling frequency of $f_s=61.44\times10^6 {\mathrm samples}/s$ or $f_s=122.88\times10^6 {\mathrm samples}/s$. $x_i(n)$ is the PSS signal itself which is of duration $N_\mathrm{PSS}=2048+144 = 2192$ samples. We will see later (after completing this lab session) that 2048 is the OFDM symbol size in samples and 144 is the cyclic prefix length.

### Initial Questions

1.  The following codeblock corresponds to the generation of the three possible PSS signals. You will see later that this is an example of the generation of a simple OFDM signal.



In [1]:
import numpy as np

def nr_pss(N_ID2: int) -> np.ndarray:
    """
    5G NR PSS per 3GPP TS 38.211 §7.4.2.2
      - Length 127, BPSK in {+1, -1}
      - m-seq (degree-7, poly x^7 + x^4 + 1), init state = 0110111
      - cyclic shift by 43*N_ID2

    Returns: float32 array of shape (127,) with values in {+1, -1}
    """
    if N_ID2 not in (0, 1, 2):
        raise ValueError("N_ID2 must be 0, 1, or 2")

    # LFSR state s[0..6], spec init: 0 1 1 0 1 1 1
    s = np.array([0,1,1,0,1,1,1], dtype=np.uint8)

    # Generate base m-sequence c(n), n=0..126
    # Fibonacci form with taps at positions 0 and 3 (x^7 + x^4 + 1):
    # output = s[0]; feedback = s[0] XOR s[3]; shift left; insert feedback at s[6]
    c = np.empty(127, dtype=np.uint8)
    for n in range(127):
        c[n] = s[0]
        fb = s[0] ^ s[3]
        s[:-1] = s[1:]
        s[-1] = fb

    # Apply cyclic shift (43*N_ID2), d(n) = c(n + 43*N_ID2 mod 127)
    shift = 43 * N_ID2
    d_bin = np.roll(c, -shift)

    # Map to BPSK: 0 -> +1, 1 -> -1
    d = (1 - 2*d_bin.astype(np.int8)).astype(np.float32)
    return d

def map_pss_to_fft_bins(pss127: np.ndarray, Nfft: int) -> np.ndarray:
    """
    Map the 127-length PSS (BPSK) onto an Nfft-sized frequency vector
    centered around DC (bin 0), using NR’s convention that includes DC.

    pss127 occupies subcarriers k = -63..+63 (total 127 tones), with
    pss127[0] -> k = -63 ... pss127[63] -> k = 0 (DC) ... pss127[126] -> k = +63.

    Returns: complex64 array of shape (Nfft,) ready for IFFT.
    """
    if len(pss127) != 127:
        raise ValueError("pss127 must be length 127")
    X = np.zeros(Nfft, dtype=np.complex64)

    k_min = -63
    for i in range(127):
        k = k_min + i                 # -63 .. +63
        bin_idx = (k % Nfft)          # wrap to 0..Nfft-1 with Python's mod
        X[bin_idx] = np.complex64(pss127[i])
    return X

# --- Build time-domain OFDM symbol and add CP (length Ncp) ---
def pss_time_domain(N_ID2: int, Nfft: int = 2048, Ncp: int = 144):
    pss = nr_pss(N_ID2)
    X = map_pss_to_fft_bins(pss, Nfft)
    x_td = np.fft.ifft(X).astype(np.complex64)      # no fftshift anywhere
    x_cp = np.concatenate([x_td[-Ncp:], x_td])      # prepend cyclic prefix
    return x_cp

# --- Generate the 3 sequences ---
x0 = pss_time_domain(0, 2048, 144)
x1 = pss_time_domain(1, 2048, 144)
x2 = pss_time_domain(2, 2048, 144)

Add some code to plot (use matplotlib) the real and imaginary components and magnitude of the one of the PSS signals. What do you see?

In [2]:
# --- Code here ---


2.  Now plot the power spectrum of the PSS signal on a dB-scale (using the numPy FFT). Estimate the bandwidth as closely as possible (in terms of physical frequencies). What do you observe about the PSS signal?



In [3]:
# --- Code here ---

3.  Plot the three auto-correlation functions and the three cross-correlation functions. To what extent can we say that these three signals are orthogonal? When using one PSS as a basis function, what is the ratio of signal energy to interference in dB if we assume that these are orthogonal?




In [4]:
# --- Code here ---

## Channel Model

The received signal is assumed to be of the form 
$$\begin{equation*}
y(n) = e^{2\pi j \Delta fn}\tilde{s}_{\mathrm{PSS},i}(n)*h(n) + z(n)
\end{equation*}$$ 
where $h(n)$ is assumed to a finite-impulse response channel which is unknown to the receiver, $z(n)$ is complex circularly-symmetric and additive white Gaussian noise, and $\Delta f$ is an unknown frequency-offset. We can assume that the channel is of the form
$$\begin{equation*}
h(n) = \sum_{l=-\lceil L/2\rceil+1}^{\lfloor L/2\rfloor}a_l\delta(n-N_f-l)
\end{equation*}$$ 
for some even number $L$. The $a_l$ are complex amplitudes *unknown to the receiver* and $N_f$ is a presentation of the unknown timing offset between the transmitter and receiver.

### Questions

1.  You have been provided several signal snapshots with different channel realizations as MATLAB signals

2.  Plot the time and frequency representations of the signals on a dB scale. What do you see?

3.  Estimate the bandwidth of the received signal? What are the signal components that are located outside the band of interest?

4.  In your opinion what contributes to the \"changing shape\" of themain signal component?

## Receiver

The primary objective is to determine the most likely $i$, or index of the transmitted PSS. In addition we would like to have the best estimate of $N_f$ and $\Delta f$ since these are required to detect the other signal components after the PSS (Lab session 2). This will be investigated in other lab sessions. There are different approaches to doing this, but here we will take the approach where $N_f$ and $\Delta f$ are discretized and are detected in a similar fashion to $i$. Let us assume that $N_f$ is discretized to the resolution of one sample.
Since the periodicity of the PSS is $N$ samples, $N_f$ can assume the values $\{0,1,\cdots,N-1\}$. Although a purely continuous random variable, we will also discretize $\Delta f$ as $\Delta f = m\Delta f_\mathrm{min},m=-\Delta f_\mathrm{max}/\Delta f_\mathrm{min},\cdots,\Delta f_\mathrm{max}/\Delta f_ \mathrm{min}$, where $\Delta f_\mathrm{max}$ is the largest frequency-offset we are likely to encounter. Under these assumptions we will consider the following detection rule for the triple $(i,N_f,m)$ 
$$\begin{equation*} (\hat{i},\hat{N_f},\hat{m}) = \underset{i,N_f,m}{\operatorname{argmax}} Y(i,N_f,m)
\end{equation*}$$ 
where $Y(i,N_f,m)$ will be chosent as the following statistic 
$$\begin{equation*}
Y(i,N_f,m) = \left|\sum_{n=0}^{N_\mathrm{PSS}} e^{-2\pi j nm\Delta f_\mathrm{min}/f_s}x_i^*(n)y(n+N_f)\right|^2
\end{equation*}$$

### Questions

1.  How is the above statistic related to the maximum-likelihood
    detector we considered in class? What is the reason for the
    difference? Think about the simpler receiver where $L=1$, $N_f=0$
    and $\Delta f=0$.

2.  What is the effect of the channel in all of this? How is it taken
    into account and what is neglected here?

3.  Show how you can use the convolution operator with the matched
    filter to implement the above maximization in a compact form (think
    of the basic receiver structures we explored in class). Plot the
    output of the three matched filters for $m=0$ (i.e. no frequency
    offset). Which PSS is most likely and try to determine $N_f$.

4.  For the most likely PSS index $i$ and $N_f$ with $m=0$, plot the
    peak value of the statistic in 100 Hz steps and a $\pm 7.5 kHz$
    window around the carrier frequency (i.e.
    $\Delta f_\mathrm{max}/\Delta f_\mathrm{min} = 75$). What is the
    most likely frequency-offset? Based on the shape of the statistic
    can you think of a efficient way to implement the frequency-offset
    estimator to obtain even finer resolution and to minimize the number
    of correlations that are required? (hint: think of a binary search
    applied to this)